In [1]:
import pandas as pd
from pathlib import Path

# Load the original dataset
raw_path = Path("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
output_path = Path("../data/processed/cleaned_churn.csv")

df = pd.read_csv(raw_path)

print("Original shape:", df.shape)

# Remove accidental spaces from column names and text values
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# Remove duplicate records, if any
df = df.drop_duplicates()

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

# Inspect missing values
print("\nMissing values before handling:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Remove rows where TotalCharges is missing
df = df.dropna(subset=["TotalCharges"]).copy()

# Save cleaned dataset
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

print("\nCleaned shape:", df.shape)
print("Missing values remaining:", df.isnull().sum().sum())
print("Saved to:", output_path)

Original shape: (7043, 21)

Missing values before handling:
TotalCharges    11
dtype: int64


C:\Users\SAI\AppData\Local\Temp\ipykernel_8052\3984776127.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:



Cleaned shape: (7032, 21)
Missing values remaining: 0
Saved to: ..\data\processed\cleaned_churn.csv


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Churn distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.tight_layout()
plt.show()

# Churn by contract
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Customer Churn by Contract Type")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Monthly charges by churn status
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges by Churn Status")
plt.tight_layout()
plt.show()

# Churn rate by tenure
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"]
)

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="TenureGroup", hue="Churn")
plt.title("Customer Churn by Tenure Group")
plt.tight_layout()
plt.show()